# 02 · Limpieza

> **Objetivo:** aplicar las reglas identificadas en el cuaderno 01 y dejar
> dos salidas: `ventas_validas.parquet` para el EDA y `cancelaciones.parquet`
> como insumo de la Unidad 3.

## Reglas (una por funcion)

| Funcion | Que hace |
|---|---|
| `quitar_duplicados` | `DataFrame.unique()` |
| `separar_cancelaciones` | parte entre `Invoice` que empiezan por `C` y el resto |
| `exigir_cliente` | descarta filas con `Customer ID` nulo |
| `filtrar_validas` | `Quantity > 0` y `Price > 0` |
| `preparar_ventas` | encadena las cuatro con `.pipe`, agrega `Importe = Quantity * Price` y registra `n_antes/n_despues` |

Secuencia esperada (verificada por los tests `-m slow`):

**1 067 371 → 1 033 036 → 1 013 932 → 779 495 → 779 425 (73,0 %)**

In [1]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from segmentacion.config import PROCESSED_DATA_DIR, RAW_DATA_DIR
from segmentacion.data.cargar import leer_parquet
from segmentacion.data.limpieza import preparar_ventas

PARQUET_CRUDO = RAW_DATA_DIR / "online_retail_II.parquet"
PARQUET_VENTAS = PROCESSED_DATA_DIR / "ventas_validas.parquet"
PARQUET_CANCELACIONES = PROCESSED_DATA_DIR / "cancelaciones.parquet"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"salida ventas       : {PARQUET_VENTAS}")
print(f"salida cancelaciones: {PARQUET_CANCELACIONES}")

salida ventas       : /Users/david.palacio/Documents/data-projects-lab/projects/00-segmentacion-clientes/data/processed/ventas_validas.parquet
salida cancelaciones: /Users/david.palacio/Documents/data-projects-lab/projects/00-segmentacion-clientes/data/processed/cancelaciones.parquet


## 1. Pipeline encadenado

`preparar_ventas` registra el conteo por regla con el logger del proyecto
(`segmentacion.utils.logger`). Si el output no muestra la linea `INFO
segmentacion.data.limpieza | pipeline de limpieza: ...`, verifica el
nivel del logger.

In [3]:
crudo = leer_parquet(PARQUET_CRUDO)
ventas, cancelaciones = preparar_ventas(crudo, log=True)
print(f"ventas: {ventas.height:,}".replace(",", " "))
print(f"cancelaciones: {cancelaciones.height:,}".replace(",", " "))

[23:16:46] INFO    segmentacion.data.limpieza | pipeline de limpieza: 1 067 371 -> sin duplicados 1 033 036 -> sin cancelaciones 1 013 932 -> con cliente 779 495 -> validas 779 425 (73.0% del original)


ventas: 779 425
cancelaciones: 19 104


## 2. Verificacion de las cifras de referencia

Si las cifras no coinciden, no editamos el cuaderno: miramos
`scripts/descargar_datos.py` (puede haber cambiado el hash del zip).

In [ ]:
assert crudo.height == 1_067_371
assert ventas.height == 779_425
# cancelaciones post-dedup (= 1 033 036 - 1 013 932)
assert cancelaciones.height == 19_104
print("cifras OK: coinciden con las cifras de referencia de la Unidad 2")

## 3. Guardar artefactos

Los Parquet son **gitignored** (se regeneran); U3 los lee por ruta.

In [5]:
ventas.write_parquet(PARQUET_VENTAS, compression="zstd")
cancelaciones.write_parquet(PARQUET_CANCELACIONES, compression="zstd")
print(f"escrito: {PARQUET_VENTAS} ({PARQUET_VENTAS.stat().st_size / 1e6:.1f} MB)")
print(f"escrito: {PARQUET_CANCELACIONES} ({PARQUET_CANCELACIONES.stat().st_size / 1e6:.1f} MB)")

escrito: /Users/david.palacio/Documents/data-projects-lab/projects/00-segmentacion-clientes/data/processed/ventas_validas.parquet (11.8 MB)
escrito: /Users/david.palacio/Documents/data-projects-lab/projects/00-segmentacion-clientes/data/processed/cancelaciones.parquet (0.3 MB)


## Preguntas de reflexion

1. `preparar_ventas` separa las cancelaciones **antes** de exigir cliente.
   ¿Por que ese orden importa? ¿Que pasaria si invirtieramos los dos pasos?
2. ¿Que tasa de devolucion aproximada tenemos (filas de cancelaciones vs.
   ventas validas)? ¿Es informacion util para U3?
3. ¿Como extenderias `filtrar_validas` para capturar un umbral de `Price`
   maximo creible (p. ej., > 10 000 £) y dejarlo trazable?

> **Siguiente paso:** `04_eda.ipynb` — exploramos distribuciones,
> temporalidad y asociaciones. Volvemos despues a RFM en `03`.